# FP8 PS2: one-shot AI research allocation
Qianshuo (Aaron) Wang and Zhenning Wang. Synthetic comparison, not a multi-slot equilibrium.
Run all cells. N=20, K=4, M=100, r=100ES, threshold=60, R=1, b=0.8v, lambda=1; seed 20260926; 1,000 independent batches, not dynamic rounds. The code runs 36 sensitivity scenarios. No external packages or downloads.


In [ ]:
from pathlib import Path
import tempfile, subprocess, sys
work = Path(tempfile.mkdtemp(prefix="ps2_"))
(work / "simulation.py").write_text('"""FP8 PS2: paired, one-shot allocation experiment (Python standard library)."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport platform\nimport random\nimport statistics as st\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nSEED = 20260926\nBATCHES = 1000\n\n\n@dataclass(frozen=True)\nclass Request:\n    pm: int\n    value: float\n    exposure: float\n    severity: float\n    arrival: int\n    shading: float = 0.8\n    budget: float = 100.0\n\n    @property\n    def risk(self):\n        return 100 * self.exposure * self.severity\n\n    @property\n    def bid(self):\n        return min(self.budget, self.shading * self.value)\n\n\ndef generate_batches(seed=SEED, count=BATCHES, alignment="independent", heterogeneous=False):\n    """Same draws across scenarios. Sorted/reversed values are alignment stress tests."""\n    rng = random.Random(seed)\n    for _ in range(count):\n        values = [100 * rng.random() for _ in range(20)]\n        exposures = [rng.random() for _ in range(20)]\n        severities = [rng.random() for _ in range(20)]\n        arrival = list(range(20))\n        rng.shuffle(arrival)\n        shade = [0.5 + 0.5 * rng.random() for _ in range(20)]\n        if alignment != "independent":\n            rank = sorted(range(20), key=lambda j: (exposures[j] * severities[j], j))\n            assigned = sorted(values, reverse=(alignment == "opposed"))\n            values = dict(zip(rank, assigned))\n        yield [Request(j + 1, values[j], exposures[j], severities[j], arrival[j],\n                       shade[j] if heterogeneous else 0.8) for j in range(20)]\n\n\ndef allocate(requests, mechanism, slots=4, reserve=1, threshold=60):\n    """Return {PM: payment}. Lower PM ID breaks all exact score/bid/arrival ties."""\n    if mechanism not in ("pure", "hybrid", "fcfs"):\n        raise ValueError("unknown mechanism")\n    if not 0 <= reserve <= slots or slots < 0:\n        raise ValueError("invalid reserve/slots")\n    if len({r.pm for r in requests}) != len(requests):\n        raise ValueError("duplicate PM")\n    if any(not (0 <= r.value <= 100 and 0 <= r.exposure <= 1 and\n                0 <= r.severity <= 1 and 0 <= r.shading <= 1 and r.budget >= 0)\n           for r in requests):\n        raise ValueError("invalid request")\n    winners = {}\n    if mechanism == "fcfs":\n        return {r.pm: 0.0 for r in sorted(requests, key=lambda r: (r.arrival, r.pm))[:slots]}\n    if mechanism == "hybrid":\n        eligible = sorted((r for r in requests if r.risk >= threshold),\n                          key=lambda r: (-r.risk, r.pm))\n        winners = {r.pm: 0.0 for r in eligible[:reserve]}\n    remaining = sorted((r for r in requests if r.pm not in winners),\n                       key=lambda r: (-r.bid, r.pm))\n    winners.update({r.pm: r.bid for r in remaining[:slots-len(winners)]})\n    return winners\n\n\ndef metrics(requests, allocation, threshold=60, weight=1.0):\n    chosen = [r for r in requests if r.pm in allocation]\n    value = sum(r.value for r in chosen)\n    risk = sum(r.risk for r in chosen)\n    payment = sum(allocation.values())\n    optimum = sum(sorted((r.value for r in requests), reverse=True)[:4])\n    eligible = sum(r.risk >= threshold for r in requests)\n    return dict(V=value, C=risk, W=value + weight*risk, payment=payment,\n                utility=value-payment, efficiency=value/optimum if optimum else None,\n                eligible_access=(sum(r.risk >= threshold for r in chosen)/eligible\n                                 if eligible else None))\n\n\ndef write_csv(path, rows):\n    rows = list(rows)\n    with path.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef mean(values):\n    values = [v for v in values if v is not None]\n    return st.mean(values) if values else None\n\n\ndef run(out="results", seed=SEED, count=BATCHES):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    batches = list(generate_batches(seed, count))\n    records, individuals = [], []\n    demo = batches[0]\n    write_csv(out/"demo_inputs.csv", [dict(**asdict(r), risk=r.risk, bid=r.bid) for r in demo])\n    for batch, requests in enumerate(batches, 1):\n        for mechanism in ("pure", "hybrid", "fcfs"):\n            allocation = allocate(requests, mechanism)\n            records.append(dict(batch=batch, mechanism=mechanism,\n                                winners=";".join(map(str, sorted(allocation))),\n                                **metrics(requests, allocation)))\n            if batch == 1:\n                for r in requests:\n                    p = allocation.get(r.pm, 0)\n                    x = int(r.pm in allocation)\n                    individuals.append(dict(mechanism=mechanism, pm=r.pm, value=r.value,\n                                            risk=r.risk, bid=r.bid, allocated=x, payment=p,\n                                            utility=x*r.value-p, credits_left=r.budget-p))\n    write_csv(out/"demo_allocations.csv", individuals)\n    write_csv(out/"primary_batches.csv", records)\n    summary = []\n    for mechanism in ("pure", "hybrid", "fcfs"):\n        subset = [r for r in records if r["mechanism"] == mechanism]\n        summary.append(dict(mechanism=mechanism, batches=count,\n                            **{k: mean(r[k] for r in subset)\n                               for k in ("V", "C", "W", "payment", "utility", "efficiency", "eligible_access")}))\n    write_csv(out/"primary_summary.csv", summary)\n    sensitivity = []\n    for alignment in ("independent", "aligned", "opposed"):\n        for hetero in (False, True):\n            sample = list(generate_batches(seed, count, alignment, hetero))\n            for threshold in (40, 60, 80):\n                for reserve in (1, 2):\n                    pairs = []\n                    for requests in sample:\n                        p = metrics(requests, allocate(requests, "pure"), threshold)\n                        h = metrics(requests, allocate(requests, "hybrid", reserve=reserve,\n                                                       threshold=threshold), threshold)\n                        pairs.append((h["V"]-p["V"], h["C"]-p["C"]))\n                    dv, dc = mean(p[0] for p in pairs), mean(p[1] for p in pairs)\n                    dw = [v+c for v,c in pairs]\n                    se = st.stdev(dw)/(count**0.5) if count > 1 else 0\n                    sensitivity.append(dict(alignment=alignment, bid_rule="heterogeneous" if hetero else "0.8v",\n                                            threshold=threshold, reserve=reserve, delta_V=dv, delta_C=dc,\n                                            delta_W_lambda0=dv, delta_W_lambda05=dv+0.5*dc,\n                                            delta_W_lambda1=dv+dc, delta_W_lambda2=dv+2*dc,\n                                            MC_SE_lambda1=se, fraction_strictly_better=mean(x>1e-10 for x in dw),\n                                            mean_break_even_lambda=(-dv/dc if dc>1e-10 else None)))\n    write_csv(out/"sensitivity.csv", sensitivity)\n    source = Path(__file__)\n    hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(out.glob("*.csv"))}\n    manifest = dict(seed=seed, batches=count, python=platform.python_version(), N=20, K=4,\n                    credits=100, threshold=60, reserve=1, welfare_weight=1,\n                    primary_bid="0.8*v", risk="100*exposure*severity",\n                    distributions="independent U[0,100] value; U[0,1] exposure and severity; random arrival",\n                    simulation_sha256=hashlib.sha256(source.read_bytes()).hexdigest(), outputs_sha256=hashes)\n    (out/"fresh_run.json").write_text(json.dumps(manifest, indent=2)+"\\n")\n    print(json.dumps(summary, indent=2))\n    print("Primary paired comparison:")\n    print(json.dumps(next(s for s in sensitivity if s["alignment"] == "independent" and\n                         s["bid_rule"] == "0.8v" and s["threshold"] == 60 and s["reserve"] == 1), indent=2))\n    return summary\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--out", default="results")\n    parser.add_argument("--seed", type=int, default=SEED)\n    parser.add_argument("--batches", type=int, default=BATCHES)\n    args = parser.parse_args()\n    if args.batches < 1:\n        parser.error("batches must be positive")\n    run(args.out, args.seed, args.batches)\n')
(work / "test_simulation.py").write_text("import unittest\nfrom simulation import Request, allocate, generate_batches, metrics\n\n\nclass MechanismTests(unittest.TestCase):\n    def test_no_eligible_returns_reserve(self):\n        requests = [Request(i, i*10, 0, 1, i) for i in range(1, 7)]\n        self.assertEqual(allocate(requests, 'pure'), allocate(requests, 'hybrid'))\n\n    def test_known_displacement_payments_and_welfare(self):\n        requests = [Request(1, 100, .1, 1, 4), Request(2, 90, .2, 1, 3),\n                    Request(3, 80, .3, 1, 2), Request(4, 70, .4, 1, 1),\n                    Request(5, 10, .9, 1, 0)]\n        pure, hybrid = allocate(requests, 'pure'), allocate(requests, 'hybrid')\n        self.assertEqual(pure, {1:80, 2:72, 3:64, 4:56})\n        self.assertEqual(hybrid, {5:0, 1:80, 2:72, 3:64})\n        p, h = metrics(requests, pure), metrics(requests, hybrid)\n        self.assertEqual(h['W']-p['W'], -10)\n        self.assertEqual(h['utility'], 64)\n\n    def test_already_winner_no_allocation_change(self):\n        requests = [Request(i, 100-i*10, 1 if i == 1 else 0, 1, i) for i in range(1, 7)]\n        pure, hybrid = allocate(requests, 'pure'), allocate(requests, 'hybrid')\n        self.assertEqual(set(pure), set(hybrid))\n        self.assertEqual(hybrid[1], 0)\n        self.assertLess(sum(hybrid.values()), sum(pure.values()))\n\n    def test_ties_boundary_and_fcfs(self):\n        requests = [Request(i, 50, .6, 1, 7-i) for i in range(1, 7)]\n        self.assertEqual(allocate(requests, 'hybrid'), {1:0, 2:40, 3:40, 4:40})\n        self.assertEqual(set(allocate(requests, 'fcfs')), {3, 4, 5, 6})\n\n    def test_budget_cap_and_invalid_duplicate(self):\n        r = Request(1, 100, 1, 1, 0, budget=20)\n        self.assertEqual(allocate([r], 'pure'), {1:20})\n        with self.assertRaises(ValueError):\n            allocate([r,r], 'pure')\n\n    def test_paired_invariants(self):\n        for requests in generate_batches(count=100):\n            for mechanism in ('pure', 'hybrid', 'fcfs'):\n                a = allocate(requests, mechanism)\n                m = metrics(requests, a)\n                self.assertEqual(len(a), 4)\n                self.assertAlmostEqual(m['utility']+m['payment'], m['V'])\n                self.assertTrue(0 <= m['efficiency'] <= 1+1e-12)\n                for r in requests:\n                    self.assertTrue(0 <= a.get(r.pm, 0) <= r.budget)\n            self.assertAlmostEqual(metrics(requests, allocate(requests, 'pure'))['efficiency'], 1)\n\n\nif __name__ == '__main__':\n    unittest.main()\n")
print("Self-contained simulation and tests are ready.")


Self-contained simulation and tests are ready.


In [ ]:
check = subprocess.run([sys.executable, "-m", "unittest", "-v", "test_simulation.py"], cwd=work, capture_output=True, text=True)
print(check.stdout + check.stderr)
check.check_returncode()


test_already_winner_no_allocation_change (test_simulation.MechanismTests.test_already_winner_no_allocation_change) ... ok
test_budget_cap_and_invalid_duplicate (test_simulation.MechanismTests.test_budget_cap_and_invalid_duplicate) ... ok
test_known_displacement_payments_and_welfare (test_simulation.MechanismTests.test_known_displacement_payments_and_welfare) ... ok
test_no_eligible_returns_reserve (test_simulation.MechanismTests.test_no_eligible_returns_reserve) ... ok
test_paired_invariants (test_simulation.MechanismTests.test_paired_invariants) ... ok
test_ties_boundary_and_fcfs (test_simulation.MechanismTests.test_ties_boundary_and_fcfs) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.008s

OK



In [ ]:
result = subprocess.run([sys.executable, "simulation.py", "--out", "results"], cwd=work, capture_output=True, text=True)
print(result.stdout)
result.check_returncode()


[
  {
    "mechanism": "pure",
    "batches": 1000,
    "V": 353.076564567162,
    "C": 101.3511739173001,
    "W": 454.4277384844621,
    "payment": 282.4612516537296,
    "utility": 70.61531291343239,
    "efficiency": 1.0,
    "eligible_access": 0.20294351630867144
  },
  {
    "mechanism": "hybrid",
    "batches": 1000,
    "V": 326.257793396918,
    "C": 137.94518492411623,
    "W": 464.2029783210342,
    "payment": 226.97709330849966,
    "utility": 99.28070008841834,
    "efficiency": 0.9242441968495722,
    "eligible_access": 0.6670445505171042
  },
  {
    "mechanism": "fcfs",
    "batches": 1000,
    "V": 199.44904768342602,
    "C": 100.42880004917477,
    "W": 299.8778477326008,
    "payment": 0.0,
    "utility": 199.44904768342602,
    "efficiency": 0.5642785872523645,
    "eligible_access": 0.20930787589498806
  }
]
Primary paired comparison:
{
  "alignment": "independent",
  "bid_rule": "0.8v",
  "threshold": 60,
  "reserve": 1,
  "delta_V": -26.81877117024399,
  "delta_

## Interpretation
The reserve increases mean W by 9.78 at lambda=1 but reduces it by 8.52 at lambda=0.5. The mean break-even weight is 0.733. Only 43.8% of batches strictly improve at lambda=1. Perfect positive rank alignment gives no allocation benefit under common shading; negative rank alignment gives mean delta W=-3.24. These are synthetic policy-index results, not avoided financial losses.
Inspect `work / "results"` for all CSV outputs. See the repository README for parameters, algorithm, assumptions, limitations and AI disclosure. Human review and peer-play evidence remain separate.
